# Data Preprocessing

This notebook is the base for :
- Exploratory Data Analysis
- Data cleaning
- Data Impuation
- Feature Engineering

## Used libraries

In [1]:
# type: ignore
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from scipy.stats import skew, kurtosis, binom, entropy
from sklearn.linear_model import LinearRegression

## Loading data

In [ ]:
x_train = pd.read_csv('../x_train.csv', index_col='ID')
y_train = pd.read_csv('../y_train.csv', index_col='ID')
train = pd.concat([x_train, y_train], axis=1)
test = pd.read_csv('../x_test.csv', index_col='ID')
train.head()

The train and test inputs are composed of 46 features.

The target of this challenge is `RET` and corresponds to the fact that the **return is in the top 50% of highest stock returns**.

Since the median is very close to 0, this information should not change much with the idea to predict the sign of the return.

## Exploratory Data Analysis

In [ ]:
print(f"The train dataset contains {x_train.shape[0]} rows and {x_train.shape[1]} columns.")
print(f"The test dataset contains {test.shape[0]} rows and {test.shape[1]} columns.")
print(f'Features in the dataset: {x_train.columns}')

The dataset is made of 46 descriptive features: (all float / int values)

- `DATE`: an index of the date (the dates are randomized and anonymized so there is no continuity or link between any dates),
- `STOCK`: an index of the stock,
- `INDUSTRY`: an index of the stock industry domain (e.g., aeronautic, IT, oil company),
- `INDUSTRY_GROUP`: an index of the group industry,
- `SUB_INDUSTRY`: a lower level index of the industry,
- `SECTOR`: an index of the work sector,
- `RET_1` to `RET_20`: the historical residual returns among the last 20 days (i.e., `RET_1` is the return of the previous day and so on),
- `VOLUME_1` to `VOLUME_20`: the historical relative volume traded among the last 20 days (i.e., `VOLUME_1` is the relative volume of the previous day and so on),
The target variable: (binary)

- `RET`: the sign of the residual stock return at time t

Let's have a look on the structure of the data. In particular, I look at missing values, the balance of the dataset and potential correlation between features of the dataset.

In [ ]:
# Plotting missing values per features
missing_values = x_train.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values.plot(kind='bar', figsize=(15, 5), title='Missing values per feature')


In [ ]:
# Plotting the distribution of the missing values per category
plt.figure(figsize=(15,10))
plt.subplots_adjust(hspace=0.5)
for i,category in enumerate(['INDUSTRY', 'INDUSTRY_GROUP', 'SECTOR', 'SUB_INDUSTRY', 'STOCK', 'DATE']): 
    plt.subplot(3,2,i+1)
    plt.title(category)
    plt.bar(train[category].sort_values().unique(),
            [(train[train[category]==sub_category].isna().sum(axis=1)>0).sum()/len(train[train[category]==sub_category])*100 for sub_category in train[category].sort_values().unique()])
    plt.xlabel('sub-category')
    plt.ylabel('%')
plt.show()

The plots show the distribution of NaN values per different type of category in percentage. We observe a "relatively" even distribution of NaN values for the categorical variable ``SECTOR``. However, the amount of NaN values for the other categorical variables appears to be less evenly distributed. It might be worthwhile to investigate if there are rows that predominantly consist of NaN values for the descriptive variables ``RET`` and ``VOLUME``. If such rows exist, we can drop them in good faith since these columns do not contribute to understanding the underlying structure. During this investigation, I noticed the following:

Given no observed returns, there is no volume observed. Therefore, we should only delete those observations where no return has been observed over the past days.

In [ ]:

ret_cols = [col for col in train.columns if 'RET_' in col]
volume_cols = [col for col in train.columns if 'VOLUME_' in col]

# describe the dataset
train[ret_cols + volume_cols].describe()

In [ ]:

# Plotting the distribution of the VOLUME features
sns.boxplot(train[[f'VOLUME_{day}' for day in range(1,21)]])
plt.ylim((-1.5,1))
plt.title('VOLUME features distribution')
plt.show()

# Plotting the distribution of the RET features
sns.boxplot(train[[f'RET_{day}' for day in range(1,21)]])
plt.ylim((-1.5,1))
plt.title('RET features distribution')
plt.show()

From the boxplots it becomes obvious that a median imputation for missing values in the colums is the better choice. (The mean is too optimistic).For the returns median and mean almost coincide. For simplicity we'll choose median imputation for both.

In [ ]:
# Target balance
plt.figure(figsize=(10,5))
plt.bar(y_train['RET'].value_counts().index, y_train['RET'].value_counts().values)
plt.title('RET target distribution')
plt.show()

# Correlation matrix with return, volume and target
ret_cols = [col for col in train.columns if 'RET' in col]
vol_cols = [col for col in train.columns if 'VOLUME' in col]
corr = train[ret_cols + vol_cols].corr().abs()
plt.figure(figsize=(10,10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation matrix with return and volume features')
plt.show()


The dataset is pretty balanced. We can therefore use a conventional approach, putting inbalance-related issues aside.
About correlation analysis : there seem to be no outlier or surprising values. We can therefore assume that apart from the NaN issue, the dataset requires only a minimum cleaning. Moreover, the original features seem to be uncorrelated.

## Data Preprocessing
- Cleaning : removing all rows with no observed returns over the past 5 days
- Imputation : Simple Impute the median for the remaining NaNs of RET_x and VOLUME_x


In [ ]:
volume_cols = [col for col in train.columns if 'VOLUME_' in col]
ret_cols = [col for col in train.columns if 'RET_' in col]

In [ ]:

# Drop all rows with too many missing returns over the past days
ret_to_drop = train[(train[ret_cols].isna().sum(axis=1)/(train[ret_cols].shape[1]) >= 0.5)][ret_cols]
train.drop(index=ret_to_drop.index, inplace=True)

In [ ]:
# Simple Impute the median for the remaining NaNs of RET_x and VOLUME_x
imputer_train = SimpleImputer(strategy='median')
imputer_test = SimpleImputer(strategy='median')
impute_cols = ret_cols + volume_cols

train[impute_cols] = imputer_train.fit_transform(train[impute_cols])
test[impute_cols] = imputer_test.fit_transform(test[impute_cols])


In [ ]:
missing_values = train.isna().sum().sum(), test.isna().sum().sum()  
missing_values

## Feature Engineering

The main drawback in this challenge would be to deal with the noise. To do that, we could create some feature that aggregate features with some statistics. 

The following cell computes statistics on a given target conditionally to some features. For example, we want to generate a feature that describe the mean of `RET_1` conditionally to the `SECTOR` and the `DATE`.

**Ideas of improvement**: change shifts, the conditional features, the statistics, and the target. 

### Sectorial Aggregation

In [ ]:

# Create new features based on the statistics by DATE aggregation for multiple shifts

shifts = [1,2,3,4,5] 
statistics = {'mean':'mean', 'std':'std','skew': lambda x: skew(x, nan_policy='omit'), 'kurt': lambda x: kurtosis(x, nan_policy='omit'), 'median':'median', 'max':'max', 'min':'min'}
target_features = ['RET','VOLUME']
for target_feature in target_features:
    tmp_name = 'DATE'
    for shift in shifts:
        for stat_name,stat in statistics.items():
            name = f'{target_feature}_{shift}_{tmp_name}_{stat_name}'
            feat = f'{target_feature}_{shift}'
            for data in [train, test]:
                data[name] = data.groupby(['DATE'])[feat].transform(stat)

In [ ]:

# Create new features based on the statistics by multiple sector and date aggregation for multiple shifts

shifts = [1,2,3,4,5] 
statistics = {'mean':'mean', 'std':'std','skew': lambda x: skew(x, nan_policy='omit'), 'kurt': lambda x: kurtosis(x, nan_policy='omit'), 'median':'median', 'max':'max', 'min':'min'}
gb_features_list = [['SECTOR', 'DATE'], ['INDUSTRY_GROUP', 'DATE'], ['INDUSTRY', 'DATE'], ['SUB_INDUSTRY', 'DATE']]
target_features = ['RET','VOLUME']
for target_feature in target_features:
    for [cat, date] in gb_features_list:
        tmp_name = cat + '_' + date
        for shift in shifts:
            for stat_name,stat in statistics.items():
                name = f'{target_feature}_{shift}_{tmp_name}_{stat_name}'
                feat = f'{target_feature}_{shift}'
                for data in [train, test]:
                    data[name] = data.groupby([cat, date])[feat].transform(stat)


In [ ]:
# Create new features based on rolling statistics for the past weeks

weeks = 4
target_features = ['RET', 'VOLUME'] 
for target_feature in target_features:
    for week in range(weeks):
        name = f'{target_feature}_WEEK_{week+1}'
        mean_name = 'mean_' + name
        std_name = 'std_' + name
        skew_name = 'skew_' + name
        # TODO : kurt_name = 'kurt_' + name
        for data in [train, test]:
            data[mean_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].mean(axis=1)
            data[std_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].std(axis=1)
            data[skew_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].skew(axis=1)
        

In [ ]:
# Normalize the mean of the return and volume features by the mean of the sector and date

shifts = [1,2,3,4] 
statistics = ['sum']
gb_features_list = [['SECTOR', 'DATE']]
target_features = ['mean_VOLUME_WEEK']
for target_feature in target_features:
    for gb_features in gb_features_list:
        tmp_name = '_'.join(gb_features)
        for shift in shifts:
            for stat in statistics:
                name = f'{target_feature}_{shift}_/total_VOLUME_SECTOR_DATE'
                feat = f'{target_feature}_{shift}'
                for data in [train, test]:
                    data[name] = data[feat]/data.groupby(gb_features)[feat].transform('sum')

shifts = [1,2,3,4] 
statistics = ['sum'] 
gb_features_list = [['SECTOR', 'DATE']]
target_features = ['mean_RET_WEEK']
for target_feature in target_features:
    for gb_features in gb_features_list:
        tmp_name = '_'.join(gb_features)
        for shift in shifts:
            for stat in statistics:
                name = f'{target_feature}_{shift}_/total_RET_of_SECTOR_DATE'
                feat = f'{target_feature}_{shift}'
                for data in [train, test]:
                    data[name] = data[feat]/data.groupby(gb_features)[feat].transform('sum')


In [ ]:
# Momentum features aggregation by sector and date

weeks = [1, 2, 3, 4]
targets = ['RET', 'VOLUME']

for target in targets:
    for week in weeks: 
        window_size = 5*week
        name = f'{target}_{window_size}_SECTOR_DATE_Momentum'
        for data in [train, test]:
            frame = data.copy()
            rolling_mean_target = frame.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(2, window_size+1)]].mean()
            target_1_mean = frame.groupby(by=['SECTOR', 'DATE'])[[f'{target}_1']].mean()
            target_1_mean_aligned, rolling_mean_target_aligned = target_1_mean.align(rolling_mean_target, axis=0, level='SECTOR')
            target_momentum = target_1_mean_aligned.sub(rolling_mean_target_aligned.mean(axis=1), axis=0)
            target_momentum.rename(columns={f'{target}_1': name},inplace=True)
            placeholder = frame.join(target_momentum, on=['SECTOR', 'DATE'], how='left')
            data[name] = placeholder[name] 

# RSI features aggregation by sector and date

targets = ["RET"]
window_size = [5, 10, 15, 20]

for window in window_size:
    name = f"RSI_{window}_SECTOR_DATE"
    for target in targets:
        for data in [train, test]:
            avg_gain_sector_day = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1, window + 1)]].mean().agg(lambda x: x[x > 0].mean(), axis=1)
            avg_loss_sector_day = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1, window + 1)]].mean().agg(lambda x: x[x < 0].mean(), axis=1).abs()
            rs_sector_day = avg_gain_sector_day / avg_loss_sector_day
            rsi_sector_date = 100 - 100 / (1 + rs_sector_day)
            data[name] = data.join(rsi_sector_date.to_frame(name), on=['SECTOR', 'DATE'], how='left')[name]


# ADL features aggregation by sector and date

window_size = [5, 10, 15, 20]
for window in window_size:
    name = f'ADL_{window}_SECTOR_DATE'
    for data in [train, test]:
        sum_adl = (data.groupby(by=["SECTOR", "DATE"])[[f'RET_{day}' for day in range(1, window + 1)]].apply(lambda x: (x > 0).sum()) - data.groupby(by=["SECTOR", "DATE"])[[f'RET_{day}' for day in range(1, window + 1)]].apply(lambda x: (x < 0).sum())).sum(axis=1)
        data[name] = data.join(sum_adl.to_frame(name), on=['SECTOR', 'DATE'], how='left')[name]


# Volatility features aggregation by sector and date

weeks = [1,2,3,4]
targets = ['RET', 'VOLUME']

for week in weeks: 
    window_size = 5*week
    for target in targets: 
        name = f'{target}_SECTOR_DATE_VOLATILITY_{window_size}'
        for data in [train, test]:
            rolling_std_target = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1,window_size+1)]].mean().std(axis=1).to_frame(name)
            placeholder = data.join(rolling_std_target, on=['SECTOR', 'DATE'], how='left')
            data[name] = placeholder[name]

### Temporal aggregation

In [3]:
def compute_moving_avg(df, cols):
    return df[cols].mean(axis=1, skipna=True)

def compute_volatility(df, cols):
    return df[cols].std(axis=1, skipna=True)

def compute_ema(df, cols, span=5):
    return df[cols].ewm(span=span, axis=1).mean().iloc[:, -1]

def compute_momentum(df, col_start, col_end):
    return df[col_end] - df[col_start]

def compute_relative_volume(df, cols, last_col):
    return df[last_col] / df[cols].median(axis=1, skipna=True)

def compute_rsi(df, cols):
    gains = df[cols].clip(lower=0).mean(axis=1, skipna=True)
    losses = df[cols].clip(upper=0).abs().mean(axis=1, skipna=True)
    return 100 - (100 / (100 + (gains / losses)))

def compute_likelihood(df, ret_cols):
    positive_counts = (df[ret_cols] > 0).sum(axis=1, skipna=True)
    n = len(ret_cols)
    p_hat = positive_counts / n  
    likelihood = binom.cdf(k=positive_counts+1, n=n+1, p=p_hat)
    
    return likelihood

def fit_ar_n_and_predict_ret(df, n):
    # Step 1: Fit AR(n) model on RET_1 = f(RET_2, ..., RET_(n+1))
    X_train = df[[f'RET_{i}' for i in range(2, n+2)]]  # Using n lags
    y_train = df['RET_1']  # Target variable

    ar_model = LinearRegression()
    ar_model.fit(X_train, y_train)

    # Step 2: Predict RET_0 using RET_1, ..., RET_n
    X_pred = df[[f'RET_{i}' for i in range(1, n+1)]].copy()  # Copy to avoid modification warnings
    X_pred.columns = X_train.columns  # Rename to match training features to avoid error

    X_pred.columns = X_train.columns 
    return ar_model.predict(X_pred)


# Z-Score Normalization
def z_score_normalization(ret_cols):
    return (ret_cols - ret_cols.mean(axis=1, keepdims=True, skipna=True)) / ret_cols.std(axis=1, keepdims=True, skipna=True)

# Hurst Exponent
def hurst_exponent(ret_cols):
    N = ret_cols.shape[1]
    Y = np.cumsum(ret_cols - ret_cols.mean(axis=1, keepdims=True, skipna=True), axis=1)
    R = np.max(Y, axis=1) - np.min(Y, axis=1)
    S = np.std(ret_cols, axis=1, skipna=True)
    return np.log(R / S) / np.log(N)

# Fractal Dimension
def fractal_dimension(ret_cols):
    N = ret_cols.shape[1]
    log_sizes = np.log(np.arange(1, N // 2))
    log_counts = []

    for size in range(1, N // 2):
        count = np.count_nonzero(np.abs(ret_cols - np.roll(ret_cols, size, axis=1)) > size, axis=1)
        log_counts.append(np.log(count))
    
    return np.polyfit(log_sizes, log_counts, 1)[0]

# Volatility Over Volume
def VoV(ret_cols, vol_cols):
    volatility = ret_cols.std(axis=1, skipna=True)  # Standard deviation of returns
    avg_volume = vol_cols.mean(axis=1, skipna=True)  # Average volume
    return volatility / avg_volume

# Volume-Return Interaction Features

# VWAR (Volume Weighted Average Return)
def vwar(ret_cols, vol_cols):
    return (ret_cols * vol_cols).sum(axis=1, skipna=True) / vol_cols.sum(axis=1, skipna=True)

# Ret/V (Return per Volume)
def ret_per_volume(ret_cols, vol_cols):
    return ret_cols / vol_cols

# Shannon Entropy of Past Returns
def shannon_entropy(ret_cols):
    entropy_values = []
    for row in ret_cols:
        _, counts = np.unique(row, return_counts=True)
        probs = counts / len(row)
        entropy_values.append(entropy(probs, base=2))
    return np.array(entropy_values)

# Permutation Entropy
def permutation_entropy(ret_cols, m=3, delay=1):
    n = ret_cols.shape[1]
    entropy_values = []
    for row in ret_cols:
        patterns = []
        for i in range(n - m * delay):
            pattern = np.argsort(row[i:i + m * delay:delay])
            patterns.append(tuple(pattern))
        
        _, counts = np.unique(patterns, return_counts=True)
        probs = counts / len(patterns)
        entropy_values.append(-np.sum(probs * np.log2(probs)))
    return np.array(entropy_values)

# Mutual Information
def mutual_information(ret_cols, vol_cols):
    mi_values = []
    for ret_row, vol_row in zip(ret_cols, vol_cols):
        mi = mutual_info_regression(ret_row.reshape(-1, 1), vol_row)
        mi_values.append(mi[0])
    return np.array(mi_values)


In [4]:

def generate_indicators(df, num_days = 20):
    ind_columns = {}

    feature_functions = {
        'MA': compute_moving_avg,
        'VOLATILITY': compute_volatility,
        'EMA': compute_ema,
        'MOMENTUM': compute_momentum,
        'REL_VOL': compute_relative_volume,
        'RSI': compute_rsi
    }
    
    for feature in ['RET', 'VOLUME']:
        cols = [f'{feature}_{i}' for i in range(1, num_days+1)]
        ind_columns[f'MA_{feature}'] = feature_functions['MA'](df, cols)
        ind_columns[f'VOLATILITY_{feature}'] = feature_functions['VOLATILITY'](df, cols)
        ind_columns[f'EMA_{feature}'] = feature_functions['EMA'](df, cols)
        ind_columns[f'MOMENTUM_{feature}'] = feature_functions['MOMENTUM'](df, f'{feature}_1', f'{feature}_{num_days}')
        if feature == 'VOLUME':
            ind_columns['REL_VOL'] = feature_functions['REL_VOL'](df, cols, f'VOLUME_{num_days}')
        else:
            ind_columns['RSI_RET'] = feature_functions['RSI'](df, cols)

    ind_columns['LIKELIHOOD_RET'] = compute_likelihood(df, [f'RET_{i}' for i in range(1, num_days+1)])
    ind_columns['RET_AR3_PRED'] = fit_ar_n_and_predict_ret(df, 3)

    ret_cols, vol_cols  = df[[f'RET_{i}' for i in range(1, num_days+1)]], df[[f'VOLUME_{i}' for i in range(1, num_days+1)]]
    
    ind_columns['Z_SCORE_RET'] = z_score_normalization(ret_cols)
    ind_columns['Z_SCORE_VOL'] = z_score_normalization(vol_cols)
    ind_columns['HURST_EXPONENT'] = hurst_exponent(ret_cols)
    ind_columns['FRACTAL_DIMENSION'] = fractal_dimension(ret_cols)
    ind_columns['VoV'] = VoV(ret_cols, vol_cols)
    ind_columns['VWAR'] = vwar(ret_cols, vol_cols)
    ind_columns['RET_PER_VOLUME'] = ret_per_volume(ret_cols, vol_cols)
    ind_columns['SHANNON_ENTROPY'] = shannon_entropy(ret_cols)
    ind_columns['PERMUTATION_ENTROPY'] = permutation_entropy(ret_cols)
    ind_columns['MUTUAL_INFORMATION'] = mutual_information(ret_cols, vol_cols)


    for feature in ['RET', 'VOLUME']:
        data = df[[f'{feature}_{i}' for i in range(1, shift+1)]]
        # statistics
        ind_columns[f'Mean_{feature}_{shift}D'] = np.mean(data, axis=1)
        ind_columns[f'Std_{feature}_{shift}D'] = np.std(data, axis=1)
        ind_columns[f'Skew_{feature}_{shift}D'] = skew(data, nan_policy='omit', axis=1)
        ind_columns[f'Kurtosis_{feature}_{shift}D'] = kurtosis(data, nan_policy='omit', axis=1)
        if feature == 'RET':
            ind_columns[f'Range_{feature}_{shift}D'] = (lambda x: np.max(x, axis=1) - np.min(x, axis=1))(data)
            ind_columns[f'Momentum_{feature}_{shift}D'] = data.iloc[:, -1] - data.iloc[:, 0]
            ind_columns[f'Cumulative_{feature}_{shift}D'] = np.prod(1 + data, axis=1) - 1
        else:
            ind_columns[f'VOL_SURGE_{shift}D'] = (df['VOLUME_1'] - ind_columns[f'Mean_VOLUME_{shift}D']) / ind_columns[f'Std_VOLUME_{shift}D']
    
    # correlation between returns and volumes
    ret, vol = df[[f'RET_{i}' for i in range(1, shift+1)]], df[[f'VOLUME_{i}' for i in range(1, shift+1)]]
    ind_columns[f'Corr_RET_VOL_{shift}D'] = [np.corrcoef(ret.iloc[i], vol.iloc[i])[0, 1] for i in range(len(ret))]

    return ind_columns

In [ ]:
df_train_indicators = pd.DataFrame(generate_indicators(train))
df_test_indicators = pd.DataFrame(generate_indicators(test))

## Data post-processing

In [ ]:
train = pd.concat([train, df_train_indicators], axis=1)
test = pd.concat([test, df_test_indicators], axis=1)

In [ ]:
infinites_train, infinites_test = train.isin([np.inf, -np.inf]).sum().sum(), test.isin([np.inf, -np.inf]).sum().sum()
infinites_train, infinites_test # Check for infinite values

In [ ]:
# infinite values
train.replace([np.inf, -np.inf], np.nan, inplace=True)
test.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
# NaN cols
nan_cols_train, nan_cols_test = train.isna().sum(), test.isna().sum()

In [ ]:
imputer_cols_test = nan_cols_test[nan_cols_test > 0].index.tolist()
imputer_cols_train = nan_cols_train[nan_cols_train > 0].index.tolist()

In [ ]:
# fill NaNs with median
imputer_train = SimpleImputer(strategy='median')
imputer_test = SimpleImputer(strategy='median')

train[imputer_cols_train] = imputer_train.fit_transform(train[imputer_cols_train])
test[imputer_cols_test] = imputer_test.fit_transform(test[imputer_cols_test])

In [ ]:
train.isnull().sum().sum(), test.isnull().sum().sum()  # Checking there is no more missing values

## Outputing the extended dataframe

In [ ]:
# Save the preprocessed data in parquet
train.to_parquet('../train_extended.parquet')
test.to_parquet('../test_extended.parquet')

Next alpha factors to try :
- [https://arxiv.org/pdf/1601.00991](101 Formulaic Alphas) 
    - https://github.com/stefan-jansen/machine-learning-for-trading/blob/main/24_alpha_factor_library/03_101_formulaic_alphas.ipynb
    - https://github.com/sumilk/algo_trading/blob/main/3_formulaic_alphas.ipynb
    - List of Alpha using rets and vols :
        - Alpha 1
        - Alpha 8
        - Alpha 14
        - Alpha 25
        - Alpha 34
        - Alpha 56
        - Alpha 2
        - Alpha 18
        - Alpha 60

- Denoising
    - Wavelet Transform feature 
        - Haar
        - Daubechies
    - FFT (Fast Fourier Transform)